# CO_pipeline



[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/Conformal_Oracle/blob/main/CO_pipeline/CO_pipeline.ipynb)



**LLMTime pipeline: text serialisation, GPT prompting, VaR/ES extraction from 1024 samples**



Published in: *Calibrating the Oracle — Distribution-Free Conformal Risk Measures for LLM-Based Financial Forecasting*



Section: Methodology


In [ ]:
# Install dependencies (if running on Google Colab)

import sys

if 'google.colab' in sys.modules:

    !pip install -q numpy pandas matplotlib scipy


In [ ]:
import numpy as np

# ── LLMTime Pipeline Overview ──
# Step 1: Serialise historical log-returns as comma-separated text
# Step 2: Prompt GPT to continue the series (1024 samples)
# Step 3: Extract VaR and ES from the empirical distribution

def extract_var_es(samples, alpha_var=0.01):
    """Extract VaR and ES from LLM simulation samples.

    Parameters
    ----------
    samples : np.ndarray, shape (N_SAMPLES,)
        Simulated log-returns from one LLM call.
    alpha_var : float
        VaR confidence level (e.g., 0.01 for 1% VaR).

    Returns
    -------
    var : float  (positive, loss convention)
    es : float   (positive, loss convention)
    """
    valid = samples[~np.isnan(samples)]
    q_lo = np.quantile(valid, alpha_var)
    var = -q_lo  # positive loss
    n_tail = max(int(len(valid) * alpha_var), 1)
    es = -np.mean(np.sort(valid)[:n_tail])
    return var, es

# ── Example ──
np.random.seed(42)
fake_samples = np.random.standard_t(df=5, size=1024) * 0.01
var, es = extract_var_es(fake_samples, alpha_var=0.01)
print(f"Example with t(5) samples:")
print(f"  VaR(1%) = {var:.6f}")
print(f"  ES(1%)  = {es:.6f}")
print(f"  ES/VaR  = {es/var:.2f}")